# K-Ride Media Server — Kaggle GPU + zrok

**노트북 B**: TTS, MusicGen, 3D Photo Inpainting, LivePortrait, FFmpeg 합성

**전제조건:**
- GPU 가속기 켜짐 (T4)
- 인터넷 접속 허용
- Kaggle Secrets: `ZROK_TOKEN`
- (선택) Dataset: `kride-media-data` — driving video, speaker wav 등

## 1. GPU 확인

In [ ]:
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("GPU가 없습니다. Settings → Accelerator → GPU T4 x2로 변경하세요.")

## 2. 의존성 설치

In [ ]:
%%bash
pip install -q \
    fastapi uvicorn[standard] python-multipart \
    coqui-tts \
    audiocraft \
    transformers accelerate \
    Pillow \
    pydantic

# FFmpeg 확인
ffmpeg -version 2>/dev/null | head -1 || echo "FFmpeg not found — apt install ffmpeg"

In [ ]:
%%bash
# FFmpeg 없으면 설치
which ffmpeg || (apt-get update -qq && apt-get install -y -qq ffmpeg)

## 3. (선택) LivePortrait 설치

인물 모션 생성이 필요한 경우에만 실행하세요. 설치에 ~5분 소요됩니다.

In [ ]:
%%bash
# LivePortrait 클론 + 의존성
if [ ! -d "/kaggle/working/LivePortrait" ]; then
    cd /kaggle/working
    git clone https://github.com/KlingAIResearch/LivePortrait.git
    cd LivePortrait
    pip install -q -r requirements.txt
    
    # 사전학습 가중치 다운로드 (~8GB)
    pip install -q -U "huggingface_hub[cli]"
    huggingface-cli download KlingTeam/LivePortrait \
        --local-dir pretrained_weights \
        --exclude "*.git*" "README.md" "docs"
    echo "LivePortrait 설치 완료"
else
    echo "LivePortrait 이미 설치됨"
fi

## 4. 서버 코드 준비

In [ ]:
import os, shutil

# Dataset에서 복사 또는 직접 확인
server_src = "/kaggle/input/kride-media-data/media_server.py"
server_dst = "/kaggle/working/media_server.py"

if os.path.exists(server_src):
    shutil.copy2(server_src, server_dst)
    print(f"서버 코드 복사: {server_dst}")
elif os.path.exists(server_dst):
    print(f"서버 코드 이미 존재: {server_dst}")
else:
    print("media_server.py가 없습니다. Dataset에 포함하거나 수동 업로드하세요.")

## 5. 모델 웜업 (선택)

미리 로딩하면 첫 요청 응답이 빨라집니다. VRAM 현황을 확인하면서 진행하세요.

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working")

# TTS 웜업 (~2-3GB VRAM)
from media_server import _get_tts
tts = _get_tts()
print(f"TTS speakers: {tts.speakers[:5] if tts.speakers else 'none'}")

print(f"\nVRAM 사용: {torch.cuda.memory_allocated(0)/1e9:.2f} GB")
print(f"VRAM 캐시: {torch.cuda.memory_reserved(0)/1e9:.2f} GB")

In [ ]:
# MusicGen 웜업 (~2-6GB VRAM 추가)
from media_server import _get_musicgen
musicgen = _get_musicgen()

print(f"\nVRAM 사용: {torch.cuda.memory_allocated(0)/1e9:.2f} GB")
print(f"VRAM 캐시: {torch.cuda.memory_reserved(0)/1e9:.2f} GB")
print(f"VRAM 여유: {(torch.cuda.get_device_properties(0).total_mem - torch.cuda.memory_reserved(0))/1e9:.1f} GB")

## 6. zrok 설치 + 터널

In [ ]:
%%bash
if [ ! -f /usr/local/bin/zrok ]; then
    curl -sL -o /tmp/zrok.tar.gz https://github.com/openziti/zrok/releases/latest/download/zrok_linux_amd64.tar.gz
    tar -xzf /tmp/zrok.tar.gz -C /tmp/
    mv /tmp/zrok /usr/local/bin/zrok
    chmod +x /usr/local/bin/zrok
    echo "zrok 설치 완료"
fi
zrok version

In [ ]:
import subprocess, os

zrok_token = os.environ.get("ZROK_TOKEN", "")
if not zrok_token:
    try:
        from kaggle_secrets import UserSecretsClient
        zrok_token = UserSecretsClient().get_secret("ZROK_TOKEN")
    except Exception:
        pass

if zrok_token:
    result = subprocess.run(["zrok", "enable", zrok_token], capture_output=True, text=True)
    if result.returncode == 0 or "already enabled" in result.stderr:
        print("zrok enable 완료")
    else:
        print(f"zrok enable: {result.stderr}")
else:
    print("ZROK_TOKEN 미설정 — Kaggle Secrets에 추가하세요")

## 7. Media Server 시작

In [ ]:
import subprocess, time, urllib.request

server_proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "media_server:app",
     "--host", "0.0.0.0", "--port", "8001", "--workers", "1"],
    cwd="/kaggle/working",
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

time.sleep(5)

try:
    with urllib.request.urlopen("http://localhost:8001/api/health") as resp:
        import json
        data = json.loads(resp.read().decode())
        print("Media Server 상태:")
        for k, v in data.items():
            print(f"  {k}: {v}")
except Exception as e:
    print(f"서버 시작 실패: {e}")

## 8. zrok 터널 시작 (포트 8001)

In [ ]:
import subprocess, time, re

zrok_proc = subprocess.Popen(
    ["zrok", "share", "public", "http://localhost:8001"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

media_url = None
for _ in range(30):
    line = zrok_proc.stdout.readline()
    if line:
        print(line.strip())
        match = re.search(r'https://[\w-]+\.share\.zrok\.io', line)
        if match:
            media_url = match.group(0)
            break
    time.sleep(1)

if media_url:
    print(f"\n{'='*60}")
    print(f"Media API URL: {media_url}")
    print(f"{'='*60}")
    print(f"\nSwagger UI: {media_url}/docs")
else:
    print("zrok URL 추출 실패")

## 9. API 테스트

In [ ]:
import requests, json, time

BASE = "http://localhost:8001"

def poll_job(job_id, timeout=600):
    """작업 완료까지 폴링"""
    start = time.time()
    while time.time() - start < timeout:
        r = requests.get(f"{BASE}/api/media/status/{job_id}")
        data = r.json()
        status = data["status"]
        print(f"  [{data['elapsed_seconds']}s] {status} — {data['progress']}")
        if status in ("completed", "failed"):
            return data
        time.sleep(3)
    return {"status": "timeout"}

print("Helper 함수 준비 완료")

In [ ]:
# 테스트 1: TTS
print("=== TTS 테스트 ===")
r = requests.post(f"{BASE}/api/media/tts", json={
    "text": "안녕하세요. 오늘은 서울의 명소를 소개해 드리겠습니다. 경복궁은 조선시대의 대표적인 궁궐입니다.",
    "language": "ko",
})
tts_job = r.json()
print(f"Job: {tts_job['job_id']}")

result = poll_job(tts_job["job_id"])
if result["status"] == "completed":
    print(f"\nTTS 완료! 파일: {result['result_path']}")
else:
    print(f"\nTTS 실패: {result.get('error', '')}")

In [ ]:
# 테스트 2: MusicGen BGM
print("=== MusicGen 테스트 ===")
r = requests.post(f"{BASE}/api/media/musicgen", json={
    "description": "calm Korean traditional ambient music, peaceful travel background",
    "duration": 10,
})
bgm_job = r.json()
print(f"Job: {bgm_job['job_id']}")

result = poll_job(bgm_job["job_id"], timeout=120)
if result["status"] == "completed":
    print(f"\nBGM 완료! 파일: {result['result_path']}")
else:
    print(f"\nBGM 실패: {result.get('error', '')}")

In [ ]:
# 테스트 3: 3D Photo Inpainting (풍경 사진 필요)
print("=== 3D Photo Inpainting 테스트 ===")

# 테스트 이미지 생성 (실제로는 풍경 사진 업로드)
from PIL import Image
import numpy as np

test_img = Image.fromarray(np.random.randint(0, 255, (512, 768, 3), dtype=np.uint8))
test_path = "/kaggle/working/test_landscape.jpg"
test_img.save(test_path)

with open(test_path, "rb") as f:
    r = requests.post(f"{BASE}/api/media/inpaint3d", files={"image": f})

inpaint_job = r.json()
print(f"Job: {inpaint_job['job_id']}")

result = poll_job(inpaint_job["job_id"], timeout=120)
print(f"결과: {result['status']}")

In [ ]:
# 테스트 4: FFmpeg 합성 (TTS + BGM + 영상)
# 위 테스트들이 모두 completed일 때 실행

print("=== FFmpeg Render 테스트 ===")
r = requests.post(f"{BASE}/api/media/render", json={
    "video_job_id": inpaint_job["job_id"],  # 3D Photo 결과
    "tts_job_id": tts_job["job_id"],         # TTS 결과
    "bgm_job_id": bgm_job["job_id"],         # BGM 결과
    "bgm_volume": 0.3,
})

render_job = r.json()
print(f"Job: {render_job['job_id']}")

result = poll_job(render_job["job_id"], timeout=120)
if result["status"] == "completed":
    print(f"\n최종 영상 완료! 파일: {result['result_path']}")
else:
    print(f"\n합성 실패: {result.get('error', '')}")

## 10. VRAM 모니터링 & 모델 교체

In [ ]:
# VRAM 현황
if torch.cuda.is_available():
    total = torch.cuda.get_device_properties(0).total_mem / 1e9
    used = torch.cuda.memory_allocated(0) / 1e9
    cached = torch.cuda.memory_reserved(0) / 1e9
    print(f"VRAM 총량: {total:.1f} GB")
    print(f"VRAM 사용: {used:.2f} GB")
    print(f"VRAM 캐시: {cached:.2f} GB")
    print(f"VRAM 여유: {total - cached:.1f} GB")

# 모델 언로드 (필요 시)
# requests.post(f"{BASE}/api/media/unload/tts")      # TTS 해제
# requests.post(f"{BASE}/api/media/unload/musicgen")  # MusicGen 해제

## 11. 종료

In [ ]:
try:
    zrok_proc.terminate()
    print("zrok 종료")
except: pass

try:
    server_proc.terminate()
    print("Media Server 종료")
except: pass